In [1]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [2]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [3]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time data, including the current price of Bitcoin. The price of Bitcoin can fluctuate rapidly and is influenced by various factors such as market demand, investor sentiment, regulatory news, and broader economic conditions.

To get the most up-to-date price, I recommend checking a reliable financial news website, cryptocurrency exchange platform, or using a financial app that tracks cryptocurrency prices. Some popular sources include CoinMarketCap, CoinGecko, and the websites of major cryptocurrency exchanges like Coinbase, Binance, and Kraken.


In [4]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "ceaf612f-1163-4b55-b853-dcca5829ba3f",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:23:20 GMT",
      "content-type": "application/json",
      "content-length": "782",
      "connection": "keep-alive",
      "x-amzn-requestid": "ceaf612f-1163-4b55-b853-dcca5829ba3f"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time data, including the current price of Bitcoin. The price of Bitcoin can fluctuate rapidly and is influenced by various factors such as market demand, investor sentiment, regulatory news, and broader economic conditions.\n\nTo get the most up-to-date price, I recommend checking a reliable financial news website, cryptocurrency exchange platform, or using a financial app that tracks cryptocurrency prices. Some popular sources include CoinMarketCap, CoinGecko, and the websites

In [5]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '66095.15000000'}


In [6]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [7]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 66095.14


In [ ]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [9]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "9e462221-ea50-4477-9598-d26f5a5c38d7",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:23:37 GMT",
      "content-type": "application/json",
      "content-length": "567",
      "connection": "keep-alive",
      "x-amzn-requestid": "9e462221-ea50-4477-9598-d26f5a5c38d7"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The User has asked for the current price of Bitcoin. The tool 'get_crypto_price' can be used to get the current price of a cryptocurrency. I need to call this tool with the symbol 'BTCUSDT' to get the current price of Bitcoin.</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_0IUjm0YXVIcJ7pbUNPkByc",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
  "

In [10]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_0IUjm0YXVIcJ7pbUNPkByc


In [11]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 66086.86


In [12]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "7570950b-8237-4a69-89ac-990fd42d008b",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:23:48 GMT",
      "content-type": "application/json",
      "content-length": "407",
      "connection": "keep-alive",
      "x-amzn-requestid": "7570950b-8237-4a69-89ac-990fd42d008b"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The tool 'get_crypto_price' has returned the result. The current price of Bitcoin (BTC) is 66086.86 USDT.</thinking>\n\n<response>The current price of Bitcoin (BTC) is 66086.86 USDT.</response>"
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 560,
    "outputTokens": 66,
    "totalTokens": 626
  },
  "metrics": {
    "latencyMs": 655
  }
}


In [13]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

<response>The current price of Bitcoin (BTC) is 66086.86 USDT.</response>


In [14]:
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "execute_python_code",
                "description": "I will execute any python code for you",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "srs": {
                                "type": "string",
                                "description": "The Python code to execute"
                            }
                        },
                        "required": ["srs"]
                    }
                }
            }
        }
    ]
}

In [15]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "79afd866-6781-4811-948c-9ff722108b1d",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:30:12 GMT",
      "content-type": "application/json",
      "content-length": "696",
      "connection": "keep-alive",
      "x-amzn-requestid": "79afd866-6781-4811-948c-9ff722108b1d"
    },
    "RetryAttempts": 1
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking> To answer this question, I need to use an external data source, which is not available in my current capabilities. However, I can provide information on the latest known price of Bitcoin. </thinking>\n\nThe latest known price of Bitcoin was around $23,000 as of October 2021. However, the price of Bitcoin can fluctuate rapidly and may have changed since then. For the most up-to-date information, I recommend checking a reliable cryptocurrency exchange or financial news website."
        }
      